In [12]:
import pandas as pd
import matplotlib.pyplot as plt 
import matplotlib.ticker as mticker 
from matplotlib.backends.backend_pdf import PdfPages

s = pd.read_csv('sales_transactions_cleaned.csv')
s['date'] = pd.to_datetime(s['date'],errors='coerce')
s['month'] = s['date'].dt.to_period('M').astype(str)
s['revenue'] =  (s['quantity']*s['price'])- pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)
display(s['revenue']) 

0         2.92
1         3.30
2         6.37
3         3.16
4         4.09
         ...  
9735     75.65
9736     96.73
9737     78.54
9738    162.83
9739     69.54
Name: revenue, Length: 9740, dtype: float64

In [13]:
m = (s.groupby('month')
    .agg(rev=('revenue','sum'),
         tx= ('transaction_id','nunique'))
    .assign(aov=lambda x: x['rev'] / x['tx'])
    .sort_index().reset_index())
display(m)

,month,rev,tx,aov
0,2023-11,62.40,10,6.240000
1,2023-12,10549.62,1350,7.814533
2,2024-01,10264.12,1364,7.525015
3,2024-02,9281.58,1175,7.899217
4,2024-03,12047.21,1244,9.684252
5,2024-04,11623.09,1199,9.693987
6,2024-05,12012.67,1177,10.206177
7,2024-06,11173.00,1103,10.129646
8,2024-07,69380.44,1092,63.535201
9,2024-08,3073.98,26,118.230000


In [14]:
t3 =(m.nlargest(3,'rev')[['month','rev']]
    .reset_index(drop=True)
    .assign(rev=lambda x:x['rev'].apply(lambda v: f'${v:,.2f}')))
display(t3)

,month,rev
0,2024-07,"$69,380.44"
1,2024-03,"$12,047.21"
2,2024-05,"$12,012.67"


In [21]:
metrices =[('rev','r','Total Sales Revenue ($)'),
           ('tx','b','Number of Transactions'),
           ('aov','g','Average order Value( $)')]

# 'b' blue น้ำเงิน
# 'g' green เขียว
# 'r' red แดง
# 'c' cyan ฟ้าอมเขียว
# 'm' magenta บานเย็น (ม่วงอมชมพู)
# 'y' yellow เหลือง
# 'k' black ดำ
# 'w' white ขาว

with PdfPages('Session1_SaledTrended5555.pdf') as pdf:

    for col, color , title in metrices:
        fig, ax = plt.subplots(figsize=(10,5))
        ax.plot(m['month'], m[col], marker='o', color=color, linewidth=2, markersize=5)
        ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
        ax.set_xlabel('Month')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}' if col != 'tx' else f'{int(v):,}'))
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches='tight'); plt.close()

    fig, ax = plt.subplots(figsize=(6,2.5))
    ax.axis('off')
    ax.set_title('Top 3 months by sales revenue',fontsize=14, fontweight='bold', pad=12)
    tbl = ax.table(cellText=t3.values, colLabels=['Month','Total Revenue'],loc='center',cellLoc='center')
    tbl.set_fontsize(12); tbl.scale(1.5,2)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()
    
print('Success')


Success
